# Imports

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt

from utils.config import OUT_DIR, CLUBS_ORDERED
from utils.graph.layout import plot_graph_by_club
SRC_ROOT = Path('').resolve().parents[1]
PARQUET_OUT_DIR = SRC_ROOT / OUT_DIR


# Dataframes loading

## Proceedings

In [ ]:
df_proceedings = pd.read_parquet(PARQUET_OUT_DIR / 'proceedings.parquet')

In [ ]:
df_proceedings.head()

## Voting

In [ ]:
df_votings = pd.read_parquet(PARQUET_OUT_DIR / 'votings.parquet')

In [ ]:
df_votings.head()

In [ ]:
df_votings.iloc[2,4]

## Votes

In [ ]:
df_votes = pd.read_parquet(PARQUET_OUT_DIR / 'votes.parquet')

In [ ]:
df_votes.head()

## Members

In [ ]:
df_members = pd.read_parquet(PARQUET_OUT_DIR / 'members.parquet')

In [ ]:
df_members.head()

In [ ]:
df_members.info()

# Graph creation

## Sanity check

In [ ]:
required_votes_cols = {"sitting", "votingNumber", "MP", "vote"}
required_votings_cols = {"sitting", "votingNumber", "yes", "totalVoted"}
required_members_cols = {"MP", "club"}

missing_votes = required_votes_cols - set(df_votes.columns)
missing_votings = required_votings_cols - set(df_votings.columns)
missing_members = required_members_cols - set(df_members.columns)

if missing_votes or missing_votings or missing_members:
    raise ValueError(
        "Missing required columns: "
        f"votes={sorted(missing_votes)}, "
        f"votings={sorted(missing_votings)}, "
        f"members={sorted(missing_members)}"
    )

## Polarity

In [ ]:

polarity_per_voting = df_votings[["sitting", "votingNumber", "yes", "totalVoted"]].copy()
polarity_per_voting = polarity_per_voting[polarity_per_voting["totalVoted"] > 0].copy()
p_yes = polarity_per_voting["yes"] / polarity_per_voting["totalVoted"]
polarity_per_voting["polarity"] = 1 - 2 * (p_yes - 0.5).abs()
polarity_per_voting.head()

## Votes with polarity df

In [ ]:

votes_with_polarity = df_votes[["sitting", "votingNumber", "MP", "vote"]].merge(
    polarity_per_voting[["sitting", "votingNumber", "polarity"]],
    on=["sitting", "votingNumber"],
    how="inner",
)
votes_with_polarity.head()

## Merge with itself

In [ ]:
pairs = votes_with_polarity.merge(
    votes_with_polarity,
    on=["sitting", "votingNumber"],
    suffixes=("_i", "_j"),
)
pairs = pairs[pairs["MP_i"] < pairs["MP_j"]].copy()
pairs.head()

## Calculate polarity-weighted agreement score

Let:
- $V$ be the set of all votings,
- $M$ be the set of all members,
- $x_{i,v} \in \{\mathrm{YES}, \mathrm{NO}, \mathrm{ABSTAIN}\}$ be the vote cast by member $i \in M$ in voting $v \in V$.

For each voting $v$, define:

$$
p_v = \frac{\mathrm{yes}_v}{\mathrm{totalVoted}_v},
\qquad
w_v = 1 - 2\left|p_v - \frac{1}{2}\right|.
$$

For each pair $(i,j)$ with $i < j$, over all votings where both members participated ($V_{ij}$):

$$
\mathrm{denom}_{ij} = \sum_{v \in V_{ij}} w_v,
$$

$$
\mathrm{score}_{ij} = \sum_{v \in V_{ij}} w_v \, \mathbf{1}\!\left(x_{i,v} = x_{j,v}\right),
$$

where $\mathbf{1}(\cdot)$ is the indicator function.

Finally, the edge weight is:

$$
\mathrm{weight}_{ij} =
\begin{cases}
\dfrac{\mathrm{score}_{ij}}{\mathrm{denom}_{ij}}, & \text{if } \mathrm{denom}_{ij} > 0, \\
\text{no edge}, & \text{if } \mathrm{denom}_{ij} = 0.
\end{cases}
$$

This matches the implementation: `denom_w = polarity`, `score_w = polarity * (vote_i == vote_j)`, then aggregation by member pair.

In [ ]:
pairs["denom_w"] = pairs["polarity_i"]
pairs["score_w"] = pairs["denom_w"] * pairs["vote_i"].eq(pairs["vote_j"]).astype(float)
pairs

In [ ]:


edge_stats = pairs.groupby(["MP_i", "MP_j"], as_index=False).agg(
    score=("score_w", "sum"),
    denom=("denom_w", "sum"),
    
) 
edge_stats = edge_stats[edge_stats["denom"] > 0].copy()
edge_stats.head()

## Calculate the weight

In [ ]:
edge_stats = edge_stats[edge_stats["denom"] > 0].copy()
edge_stats["weight"] = edge_stats["score"] / edge_stats["denom"]
edge_stats

## Add members' clubs to the graph

In [ ]:
members_nodes = df_members[["MP", "club"]].drop_duplicates(subset=["MP"]).copy()

In [ ]:


G = nx.Graph()
for row in members_nodes.itertuples(index=False):
    G.add_node(int(row.MP), club=row.club)

In [ ]:
for row in edge_stats.itertuples(index=False):
    G.add_edge(
        int(row.MP_i),
        int(row.MP_j),
        weight=float(row.weight)
    )

graph_stats = {
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "weight_min": float(edge_stats["weight"].min()) if not edge_stats.empty else None,
    "weight_mean": float(edge_stats["weight"].mean()) if not edge_stats.empty else None,
    "weight_max": float(edge_stats["weight"].max()) if not edge_stats.empty else None,
}

graph_stats

# Visualizing the weights

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(edge_stats["weight"], bins=30, kde=True)
plt.title("Histogram of Edge Weights")
plt.xlabel("Weight")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Sparsing the graph with threshold

In [ ]:
THRESHOLD_DOWN = 0.2
THRESHOLD_UP = 1.0

sparse_G = nx.Graph(
    (u, v, d) for u, v, d in G.edges(data=True) if THRESHOLD_DOWN <= d["weight"] <= THRESHOLD_UP
)
for node in G.nodes:
    if node in G.nodes:
        nx.set_node_attributes(sparse_G, {node: G.nodes[node]})

In [ ]:
plot_graph_by_club(
    sparse_G,
    clubs_ordered=CLUBS_ORDERED,
    node_size=240,
    jitter=0.08,
    
    seed=7,
    show_node_ids=True,
    node_id_font_size=8,
    title=f"MP Voting Similarity Graph (edges with weight in [{THRESHOLD_DOWN}, {THRESHOLD_UP}])",
)


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(edge_stats["weight"], bins=30, kde=True)
plt.title("Histogram of Edge Weights")
plt.xlabel("Weight")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
df_members.loc[df_members["MP"] == 309]